# Comparing spatial clusters across time

**Question:** When observations are split into temporal windows, what can a change in spatial cluster patterns tell us—and what can't it establish?

**Learning objectives:** compare samples under consistent coordinate and parameter choices, identify DBSCAN clusters and noise, and consider unequal sample sizes and independent cluster labels. **Related Part 1:** spatial clustering and visualising clusters.

This is a synthetic spatial-temporal demonstration, **not** a replication of observed Queenstown Citydata points. The legacy course application remains available for its separately verified data source. No Google Drive is needed.

## 1. Generate two time windows

Coordinates are in a fictional planar metre-based system. The dataset is generated with a fixed seed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
rng=np.random.default_rng(2610)
windows={"morning":[((200,200),55,90),((750,700),85,110)],
         "afternoon":[((300,680),70,95),((790,300),65,80)]}
records=[]
for period,groups in windows.items():
    for centre,spread,n in groups:
        for x,y in rng.normal(centre,spread,(n,2)):
            records.append((period,x,y))
data=pd.DataFrame(records,columns=["period","x_m","y_m"])
print(data.groupby("period").size())

## 2. Keep scale and parameters constant

DBSCAN uses distances in the supplied coordinate units. Run each period independently and retain a common epsilon and plotting extent; cluster label 0 in one period does not mean the same group as label 0 in another.

In [ ]:
summary=[]
fig,axes=plt.subplots(1,2,figsize=(10,4),sharex=True,sharey=True)
for ax,(period,part) in zip(axes,data.groupby("period",sort=False)):
    fit=DBSCAN(eps=95,min_samples=6).fit(part[["x_m","y_m"]])
    labels=fit.labels_
    summary.append((period,len(part),len(set(labels)-{-1}),int((labels==-1).sum())))
    ax.scatter(part["x_m"],part["y_m"],c=labels,s=9)
    ax.set(title=period,xlim=(0,1000),ylim=(0,1000),xlabel="x (m)",ylabel="y (m)")
plt.tight_layout()
plt.show()
print(pd.DataFrame(summary,columns=["period","n","clusters","noise"]))

## 3. Guided investigation

**TODO (interpretation):** Change `eps` to 55 or 140 and rerun the analysis. Which patterns persist, and which classifications change? Why might a different observation volume across periods affect clustering results?

**TODO (extension):** Can you legitimately match clusters across time by label number alone? Suggest an appropriate spatial overlap or centroid-distance comparison, together with its limitations.

## Interpretation

A cluster in this analysis means a neighbourhood of observed coordinate points under a particular distance/minimum-density rule. It is not evidence that individuals travelled together, nor does it establish a causal change in urban activity between times.